# Fine-tune E5 and the cross-encoder on ESCI (Colab GPU)

Part of [bilingual-product-search](../README.md). The laptop runs PyTorch on 2 CPU threads, so training happens
here and evaluation happens back on the laptop with `scripts/run_search.py`, on exactly the same 1,000 test
queries per language and 50k-product corpora as the off-the-shelf results.

**What is trained**
1. `intfloat/multilingual-e5-small` (bi-encoder): contrastive loss with in-batch negatives plus one hard
   negative per query, drawn from that query's products labelled Irrelevant or Substitute.
2. `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (reranker): pointwise binary cross-entropy on every judged
   (query, product) pair, with the ESCI gains as soft targets (Exact 1, Substitute 0.1, Complement 0.01,
   Irrelevant 0), the same gains NDCG uses.

**Data** comes from `scripts/make_finetune_data.py` on the laptop: ESCI `small_version`, jp + us, train split
only, minus the 300 dev queries per language that `run_search.py` uses to tune the hybrid weight. Test queries
are in the test split, and the script checks no training query shares its text with a test query.
Hyperparameters are fixed up front, not tuned on the test set. 300 training queries per language are held back
here only as a sanity check that training helps.

**Before running**
1. On the laptop: `python scripts/make_finetune_data.py` and upload `data/finetune/esci_train_pairs.parquet`
   (about 140 MB) to Google Drive as `MyDrive/bps/esci_train_pairs.parquet`.
2. Runtime → Change runtime type → T4 GPU (or better).
3. Run all cells. About 30 to 40 minutes on a T4. Models are saved to `MyDrive/bps/models/`.

In [ ]:
# same library versions as the laptop (requirements.txt), so the saved models load there unchanged
!pip -q install sentence-transformers==3.3.1 transformers==4.46.3

In [ ]:
import os, time, math, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/bps")
except ImportError:  # local smoke test
    BASE = Path(os.environ.get("BPS_BASE", "data/finetune"))

SMOKE = os.environ.get("BPS_SMOKE") == "1"   # tiny run on CPU to check the code, never for real results
DEV = "cuda" if torch.cuda.is_available() else "cpu"
assert DEV == "cuda" or SMOKE, "No GPU: Runtime -> Change runtime type -> T4 GPU"
if DEV == "cuda":
    print(torch.cuda.get_device_name(0))
OUT = BASE / "models"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

E5 = "intfloat/multilingual-e5-small"
RERANKER = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
GAIN = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}

# fixed hyperparameters (common defaults for these model sizes; not tuned on the evaluation queries)
BI = dict(epochs=2, batch_size=128, lr=3e-5, warmup=0.1, max_len=64, scale=20.0, max_pos_per_query=4)
CE = dict(epochs=1, batch_size=64, lr=2e-5, warmup=0.1, max_len=128)
N_SANITY = 300   # held-back training queries per locale, for the before/after sanity check only
if SMOKE:
    BI.update(epochs=1, batch_size=8)
    CE.update(batch_size=8)
    N_SANITY = 5

## Data

In [ ]:
pairs = pd.read_parquet(BASE / "esci_train_pairs.parquet")
if SMOKE:
    keep = pairs.drop_duplicates("query_id").groupby("locale").head(20)["query_id"]
    pairs = pairs[pairs["query_id"].isin(keep)]
print(pairs.groupby("locale").agg(queries=("query_id", "nunique"), pairs=("doc", "size")))

# hold back a few training queries per locale as a sanity check (never used for training or model choice)
rng = np.random.default_rng(SEED)
sanity_ids = set()
for loc, g in pairs.groupby("locale"):
    with_exact = np.sort(g.loc[g["esci_label"] == "E", "query_id"].unique())
    sanity_ids |= set(rng.choice(with_exact, size=min(N_SANITY, len(with_exact)), replace=False).tolist())
sanity = pairs[pairs["query_id"].isin(sanity_ids)]
train = pairs[~pairs["query_id"].isin(sanity_ids)]
print(f"train: {train['query_id'].nunique():,} queries, {len(train):,} pairs; "
      f"sanity: {sanity['query_id'].nunique()} queries")

In [ ]:
def ndcg10(scores, gains):
    order = np.argsort(-np.asarray(scores), kind="stable")[:10]
    g = np.asarray(gains, float)
    disc = 1 / np.log2(np.arange(2, 12))
    ideal = (np.sort(g)[::-1][:10] * disc[: min(10, len(g))]).sum()
    return float((g[order] * disc[: len(order)]).sum() / ideal) if ideal > 0 else float("nan")


def sanity_ndcg(score_fn):
    # rerank each held-back query's judged products (the ESCI ranking task); mean NDCG@10 per locale
    out = {}
    for loc, g in sanity.groupby("locale"):
        vals = [ndcg10(score_fn(q["query"].iloc[0], q["doc"].tolist()), q["gain"].tolist())
                for _, q in g.groupby("query_id")]
        out[loc] = round(float(np.nanmean(vals)), 4)
    return out


def linear_warmup(opt, total, warmup):
    w = max(1, int(total * warmup))
    return torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / w, max(0.0, (total - s) / max(1, total - w))))

## 1. Bi-encoder: multilingual E5-small

Each batch holds queries from one language. Every query brings one Exact product (positive) and one
Irrelevant or Substitute product of the same query (hard negative); the other queries' positives and negatives
in the batch act as extra negatives. A query appears at most once per batch, so its own other Exact products
are never used as negatives. Each epoch cycles through up to 4 Exact products per query.

In [ ]:
from sentence_transformers import SentenceTransformer

queries = []
for (loc, qid), g in train.groupby(["locale", "query_id"]):
    pos = g.loc[g["esci_label"] == "E", "doc"].tolist()
    neg = g.loc[g["esci_label"].isin(["I", "S"]), "doc"].tolist()
    if pos:
        queries.append((loc, g["query"].iloc[0], pos, neg))
docs_by_loc = {loc: g["doc"].unique() for loc, g in train.groupby("locale")}
print(f"{len(queries):,} queries with an Exact product; "
      f"{np.mean([len(q[3]) == 0 for q in queries]):.1%} have no I/S product (they get a random same-language one)")


def epoch_batches(rng, bs, max_pos):
    rounds = {}
    for loc, q, pos, neg in queries:
        for r, p in enumerate(rng.permutation(pos)[:max_pos]):
            n = rng.choice(neg) if neg else rng.choice(docs_by_loc[loc])
            rounds.setdefault((r, loc), []).append(("query: " + q, "passage: " + p, "passage: " + n))
    batches = []
    for items in rounds.values():   # one round = each query at most once, so no duplicates in a batch
        idx = rng.permutation(len(items))
        batches += [[items[i] for i in idx[s:s + bs]] for s in range(0, len(idx), bs) if len(idx[s:s + bs]) > 1]
    return [batches[i] for i in rng.permutation(len(batches))]


bi = SentenceTransformer(E5, device=DEV)
bi.max_seq_length = BI["max_len"]


def bi_scores(model):
    def fn(q, docs):
        e = model.encode(["query: " + q] + ["passage: " + d for d in docs], normalize_embeddings=True,
                         convert_to_tensor=True, show_progress_bar=False)
        return (e[1:] @ e[0]).cpu().numpy()
    return fn


bi_before = sanity_ndcg(bi_scores(bi))
print("sanity NDCG@10 before:", bi_before)

In [ ]:
def embed(model, texts):
    feats = model.tokenize(texts)
    feats = {k: v.to(DEV) for k, v in feats.items()}
    return F.normalize(model(feats)["sentence_embedding"], dim=-1)


n_steps = sum(len(epoch_batches(np.random.default_rng(e), BI["batch_size"], BI["max_pos_per_query"]))
              for e in range(BI["epochs"]))
opt = torch.optim.AdamW(bi.parameters(), lr=BI["lr"], weight_decay=0.01)
sched = linear_warmup(opt, n_steps, BI["warmup"])
scaler = torch.amp.GradScaler(enabled=DEV == "cuda")
bi.train()
step, t0 = 0, time.time()
for epoch in range(BI["epochs"]):
    for batch in epoch_batches(np.random.default_rng(epoch), BI["batch_size"], BI["max_pos_per_query"]):
        q, p, n = zip(*batch)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            eq, ed = embed(bi, list(q)), embed(bi, list(p) + list(n))
            logits = (eq @ ed.T).float() * BI["scale"]     # B x 2B: positives on the diagonal
            loss = F.cross_entropy(logits, torch.arange(len(q), device=DEV))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(bi.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        step += 1
        if step % 100 == 0 or step == n_steps:
            print(f"epoch {epoch} step {step}/{n_steps} loss {loss.item():.3f} ({time.time() - t0:.0f}s)", flush=True)
bi.eval()

bi_after = sanity_ndcg(bi_scores(bi))
print("sanity NDCG@10 before:", bi_before, "after:", bi_after)
bi_dir = OUT / "e5-small-esci"
bi.save(str(bi_dir))
print("saved", bi_dir)

## 2. Cross-encoder reranker

Every judged pair of the training queries, shuffled, with the ESCI gain as a soft BCE target.

In [ ]:
from sentence_transformers import CrossEncoder

ce = CrossEncoder(RERANKER, num_labels=1, max_length=CE["max_len"], device=DEV)


def ce_scores(model):
    return lambda q, docs: np.asarray(model.predict([(q, d) for d in docs], batch_size=64, show_progress_bar=False))


ce_before = sanity_ndcg(ce_scores(ce))
print("sanity NDCG@10 before:", ce_before)

In [ ]:
net, tok = ce.model, ce.tokenizer
data_q, data_d, data_y = train["query"].to_numpy(), train["doc"].to_numpy(), train["gain"].to_numpy(np.float32)
bs = CE["batch_size"]
n_steps = CE["epochs"] * math.ceil(len(data_y) / bs)
opt = torch.optim.AdamW(net.parameters(), lr=CE["lr"], weight_decay=0.01)
sched = linear_warmup(opt, n_steps, CE["warmup"])
scaler = torch.amp.GradScaler(enabled=DEV == "cuda")
net.train()
step, t0 = 0, time.time()
for epoch in range(CE["epochs"]):
    order = np.random.default_rng(100 + epoch).permutation(len(data_y))
    for s in range(0, len(order), bs):
        i = order[s:s + bs]
        feats = tok(list(data_q[i]), list(data_d[i]), truncation=True, max_length=CE["max_len"],
                    padding=True, return_tensors="pt").to(DEV)
        y = torch.from_numpy(data_y[i]).to(DEV)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            logits = net(**feats).logits.squeeze(-1).float()
            loss = F.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        step += 1
        if step % 500 == 0 or step == n_steps:
            print(f"epoch {epoch} step {step}/{n_steps} loss {loss.item():.4f} ({time.time() - t0:.0f}s)", flush=True)
net.eval()

ce_after = sanity_ndcg(ce_scores(ce))
print("sanity NDCG@10 before:", ce_before, "after:", ce_after)
ce_dir = OUT / "mmarco-minilm-esci"
ce.save(str(ce_dir))
print("saved", ce_dir)

In [ ]:
import json
log = {"bi_encoder": {"base": E5, "hparams": BI, "sanity_ndcg10_before": bi_before, "sanity_ndcg10_after": bi_after},
       "cross_encoder": {"base": RERANKER, "hparams": CE, "sanity_ndcg10_before": ce_before, "sanity_ndcg10_after": ce_after},
       "train_queries": int(train["query_id"].nunique()), "train_pairs": int(len(train)),
       "sanity_queries": int(sanity["query_id"].nunique()), "device": torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu",
       "smoke": SMOKE}
(OUT / "finetune_log.json").write_text(json.dumps(log, indent=2))
print(json.dumps(log, indent=2))

## Back on the laptop

Download `MyDrive/bps/models/` (the two model folders and `finetune_log.json`) into
`bilingual-product-search/models/` (git-ignored), then:

```bash
python scripts/run_search.py --encoder models/e5-small-esci --reranker models/mmarco-minilm-esci --tag finetuned
python scripts/compare_runs.py --new finetuned
```

The first command re-encodes both 50k corpora with the fine-tuned E5 (about 40 minutes on the laptop) and
reranks with the fine-tuned cross-encoder (about 50 minutes). The second prints the before/after table with
95% intervals and paired p-values against `outputs/search/results.json`.